# Anatomy-initialized repeated-structure curve

## Goal checkpoint

Recover one ordered canonical PT object from many unordered cross-sectional observations, with unknown individual nephron identities integrated into variation around the mean path. This is a mean-atlas special case: it does not track physical nephrons or identify longitudinal length. Test whether weak anatomical order changes canonical curve construction usefully, rather than adding another probability-flow or response projector.

Compare source-only and pooled curves initialized with an S1→S2→S3 polyline against their generic-initialization scFates counterparts. The three nodes are centroids of source modeling features; no canonical marker genes, expected expression curves or fixed segment widths are supplied. Subsequent elastic fitting can move these nodes, so initialization does not enforce final anatomical order. DPT stays visible as a reference.


## Frozen actual-data protocol

Two healthy controls, three marker-excluded gene folds, source-only 15-component count representation (dispersion 100), 30 curve nodes, elastic lambda 0.01 and mu 0.1, seed 15, and the same 15-neighbor extension. Source labels initialize/root/orient; every query calibration label is −1. Full and S3-only calibration use identical hash-defined evaluation rows. S3 labels select the stress population, then are hidden from fitting. Anatomy initialization uses additional source information relative to the baseline's root/orientation only; the weaker endpoint ablation is reserved for a successful result.

Original training coordinates fit source-only response splines (ridge 0.001, grid 101). Compare point and assignment-averaged response reduction for every curve and graph arm. Query labels enter privileged segment/coverage oracles and evaluation, never molecular projection. Thin the full query count matrix once per fold, recompute all-gene exposure, and freeze every fit. Record source segment medians, all attempts, failures, calibration sensitivity and gene-panel agreement. No covariance, offset, warping, specimen weights, occupancy matching or parameter sweep is added.

All-gene exposure includes held-out responses and panels were selected in earlier development; neither is independent fine anatomical truth. Assignment weights are uncalibrated, gene folds are not biological replicates, and marker annotations contributed upstream source information. Molecular stability cannot establish actual within-segment anatomical order. Depth is excluded; injury, human and TF branches remain closed.


In [ ]:
from pathlib import Path
import argparse,os,sys,hashlib,io,json
from importlib.metadata import version
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.trajectory_benchmark import fit_scfates_axis,fit_dpt_axis,project_neighbor_axis
from pseudospace.repeated_structure import fit_atlas
from pseudospace.within_segment_validation import fit_segment_covariate_oracle,predict_segment_covariate_oracle
from pseudospace.stage_cache import cached_payload
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'repeated_structure_anatomy_initialization';OUT.mkdir(parents=True,exist_ok=True)
NOTEBOOK_LOGIC_VERSION='28.anatomy_initialization.1'
CONTROLS=('Ctrl1A2','Ctrl1A4');SEED,N_COMPONENTS,N_FOLDS=15,15,3
GRID=np.linspace(0,1,101)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
VERSIONS={name:version(name) for name in ['scanpy','scFates','anndata','scikit-learn','elpigraph-python']}
CODE_FILES=[REPO/'pseudospace'/name for name in ['repeated_inputs.py','count_representation.py','trajectory_benchmark.py','repeated_structure.py','within_segment_validation.py','stage_cache.py']]
CODE_DIGEST=hashlib.sha256(b''.join(path.read_bytes() for path in CODE_FILES)).hexdigest()
LEGACY27_OUT=RESULTS_ROOT/'repeated_structure_assignment_prediction'
LEGACY25_OUT=RESULTS_ROOT/'groupwise_repeated_structure_reconstruction'
LEGACY27_FILES={name:LEGACY27_OUT/(name+'.csv') for name in ['scores','thin_scores','selected_scores','positions']}
LEGACY25_FILES={name:LEGACY25_OUT/(name+'.csv') for name in ['scores','positions','selected_scores']}
if any(not p.is_file() for p in [*LEGACY27_FILES.values(),*LEGACY25_FILES.values()]):
    raise FileNotFoundError('Completed notebook-25/27 references required')
legacy27={name:pd.read_csv(path) for name,path in LEGACY27_FILES.items()}
legacy25={name:pd.read_csv(path) for name,path in LEGACY25_FILES.items()}
PANEL_PATH=RESULTS_ROOT/'within_segment_repeated_structure_validation'/'selected_genes.csv'
frozen_panel=pd.read_csv(PANEL_PATH,usecols=['source_specimen','test_specimen','fold','segment','gene','training_cv_gain'])
if frozen_panel.duplicated(['source_specimen','test_specimen','fold','segment','gene']).any():
    raise ValueError('Duplicated frozen program gene')
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,components=N_COMPONENTS,
 folds=N_FOLDS,markers=sorted(MARKERS),count_theta=100.,graph=dict(nodes=30,elastic_lambda=.01,elastic_mu=.1,dpt_neighbors=30),
 initialization=['default','source_anchor_centroids_S1_S2_S3'],initial_edges=[[0,1],[1,2]],
 calibration_hash='sha256(structure_id) first8 %2==0',evaluation_hash='complement',regimes=['full','S3_only'],
 query_anatomy='hidden_minus_one',projection_neighbors=15,response_ridge=.001,response_grid=GRID.tolist(),
 response_reduction=['point','assignment_average'],quarter_read_fraction=.25,quarter_read_seed_rule='seed+fold',
 oracle=dict(degree=2,ridge=.01),panel_digest=hashlib.sha256(PANEL_PATH.read_bytes()).hexdigest(),versions=VERSIONS,
 legacy27_digests={k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in LEGACY27_FILES.items()},
 legacy25_digests={k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in LEGACY25_FILES.items()},depth_used=False)
print('Installed versions:',VERSIONS)


## Validated actual healthy inputs

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly two healthy controls')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']
X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0):
    raise ValueError('Invalid expression or all-gene exposure')
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,
                code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))


In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0)
                               for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,prediction in predictions.items():
        scaled=(response-prediction)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any():
                out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),
                                normalized_mse=float(scaled[use].mean())))
    return out

def selected_rows(response,predictions,train_response,train_anatomy,eval_anatomy,panel,gene_to_j,meta):
    out=[]
    for seg,name in enumerate(['S1','S2','S3']):
        selected=panel[panel.segment==name]
        if selected.empty or not set(selected.gene).issubset(gene_to_j):
            raise ValueError('Invalid frozen panel for '+name)
        jj=np.array([gene_to_j[g] for g in selected.gene],int)
        variance=np.maximum(np.var(train_response[train_anatomy==seg],axis=0),1e-3)[jj]
        use=eval_anatomy==seg
        if not use.any(): raise ValueError('Missing evaluation segment '+name)
        for method,prediction in predictions.items():
            out.append(dict(**meta,method=method,segment=name+'_selected',n=int(use.sum()),
              selected_gene_count=len(jj),normalized_mse=float(np.mean(
               (response[use][:,jj]-prediction[use][:,jj])**2/variance))))
    return out


def assignment_predictions(assignments,decoders):
    out={}
    for method,a in assignments.items():
        out[method]=decoders[method].predict(a['z_mean'])
        out[method+'_assignment']=decoders[method].predict_distribution(a['support'],a['weights'])
    return out


## Source and pooled curve construction

In [ ]:
def run_anatomy_initialization():
    names = ['scores', 'thin_scores', 'selected_scores', 'thin_selected_scores',
             'stability', 'positions', 'diagnostics', 'failures',
             'source_reference_scores', 'source_reference_thin_scores',
             'source_reference_selected_scores', 'source_reference_positions']
    frames = {name: [] for name in names}
    graph_methods = [('source_scFates_neighbor', 'scFates'),
                     ('source_DPT_neighbor', 'DPT'),
                     ('source_anatomy_curve_neighbor', 'anatomy')]

    def add_diagnostic(key, stage, regime, method, status, metadata=None, error=None,
                       z=None, source_anatomy=None):
        record = dict(**key, stage=stage, regime=regime, method=method,
                      status=status, metadata=json.dumps(metadata or {}, sort_keys=True))
        if error is not None:
            record['error'] = str(error)
        if z is not None:
            medians = [float(np.median(z[source_anatomy == label]))
                       for label in (0, 1, 2)]
            record['source_segment_medians'] = json.dumps(medians)
            record['source_medians_ordered'] = bool(medians[0] < medians[1] < medians[2])
        frames['diagnostics'].append(record)


    for source in CONTROLS:
        target = next(s for s in CONTROLS if s != source)
        tr, te = specimens == source, specimens == target
        query_ids = ids[te]
        calibration = np.array([
            int(hashlib.sha256(x.encode()).hexdigest()[:8], 16) % 2 == 0
            for x in query_ids
        ])
        evaluation = ~calibration
        for fold in range(N_FOLDS):
            key = dict(source_specimen=source, test_specimen=target, fold=fold)
            # One source-only feature fit and one paired thinning draw per fold.
            try:
                mg = np.flatnonzero(~marker_mask & (gene_fold != fold))
                eg = np.flatnonzero(~marker_mask & (gene_fold == fold))
                C = dense_panel(raw_counts, tr, mg)
                Q = dense_panel(raw_counts, te, mg)
                Y, transform = fit_count_representation(
                    C, raw_total[tr], specimens[tr], N_COMPONENTS, theta=100., seed=SEED)
                qY = transform_count_representation(Q, raw_total[te], transform)
                sa = anatomy[tr]
                response, qresponse = X[tr][:, eg], X[te][:, eg]
                variance = normalized_variance(response, specimens[tr])
                panel = frozen_panel[(frozen_panel.source_specimen == source) &
                                     (frozen_panel.test_specimen == target) &
                                     (frozen_panel.fold == fold)]
                gene_to_j = {g: j for j, g in enumerate(genes[eg])}
                full = sparse.csr_matrix(raw_counts[te])
                rng = np.random.default_rng(SEED + fold)
                thin = sparse.csr_matrix((
                    rng.binomial(full.data.astype(np.int64), .25),
                    full.indices.copy(), full.indptr.copy()), shape=full.shape)
                thinlib = np.asarray(thin.sum(axis=1)).ravel().astype(float)
                if np.any(thinlib <= 0):
                    raise ValueError('Quarter-read thinning produced zero exposure')
                thinC = dense_panel(thin, np.ones(len(thinlib), dtype=bool), mg)
                thin_qY = transform_count_representation(thinC, thinlib, transform)
                thin_qY_eval = thin_qY[evaluation]
                qY_eval = qY[evaluation]
                eval_response, eval_a = qresponse[evaluation], anatomy[te][evaluation]
                eval_ids = query_ids[evaluation]
                detected_train = np.asarray((raw_counts[tr][:, mg] > 0).sum(axis=1)).ravel()
                detected_query = np.asarray((raw_counts[te][:, mg] > 0).sum(axis=1)).ravel()
                train_cov = np.column_stack([np.log(raw_total[tr]), np.log1p(n_spots[tr]),
                                             np.log1p(detected_train)])
                query_cov = np.column_stack([np.log(raw_total[te]), np.log1p(n_spots[te]),
                                             np.log1p(detected_query)])
                coverage_model = fit_segment_covariate_oracle(
                    response, train_cov, sa, specimens[tr], ridge=.01, degree=2)
                coverage_pred_all = predict_segment_covariate_oracle(
                    coverage_model, query_cov, anatomy[te])
                segment_means = {a: response[sa == a].mean(axis=0) for a in (0, 1, 2)}
                segment_pred_all = np.vstack([segment_means[int(a)] for a in anatomy[te]])
            except Exception as exc:
                frames['failures'].append(dict(**key, stage='sourcefold_setup', regime='',
                                               method='sourcefold_setup', error=str(exc)))
                add_diagnostic(key, 'source', '', 'sourcefold_setup', 'failed', error=exc)
                print('Failed:', source, 'to', target, 'fold', fold, str(exc), flush=True)
                continue

            # Source-only axes and decoders are fitted once, from original counts.
            source_axes, source_decoders, source_assignments, source_thin_assignments = {}, {}, {}, {}
            source_anatomy = sa
            for method_name, algorithm in graph_methods:
                init = 'anatomy' if algorithm == 'anatomy' else 'default'
                try:
                    if algorithm == 'DPT':
                        z, meta = fit_dpt_axis(Y, sa, n_neighbors=30, seed=SEED)
                    else:
                        z, meta = fit_scfates_axis(Y, sa, nodes=30, seed=SEED,
                                                   initialization=init)
                    decoder = fit_gene_spline(response, z, specimens[tr])
                    alias = method_name.replace('source_', '')
                    assignment = project_neighbor_axis(Y, z, qY, k=15, return_distribution=True)
                    thin_assignment = project_neighbor_axis(Y, z, thin_qY, k=15, return_distribution=True)
                    source_axes[method_name] = z
                    source_decoders[method_name] = decoder
                    source_assignments[alias] = assignment
                    source_thin_assignments[alias] = thin_assignment
                    add_diagnostic(key, 'source', '', method_name, 'complete', metadata=meta,
                                   z=z, source_anatomy=source_anatomy)
                except Exception as exc:
                    frames['failures'].append(dict(**key, stage='source', regime='',
                                                   method=method_name, error=str(exc)))
                    add_diagnostic(key, 'source', '', method_name, 'failed', error=exc)
                    print('Failed:', source, 'to', target, 'fold', fold, method_name, str(exc), flush=True)

            # Full-query source references reproduce both frozen predecessor workflows.
            ref_names = {name: name.replace('source_', '') for name in source_axes
                         if name in ('source_scFates_neighbor', 'source_DPT_neighbor')}
            ref_assignments = {ref_names[name]: source_assignments[ref_names[name]]
                               for name in ref_names}
            ref_decoders = {ref_names[name]: source_decoders[name] for name in ref_names}
            ref_pred = assignment_predictions(ref_assignments, ref_decoders) if ref_assignments else {}
            ref_meta = dict(**key)
            frames['source_reference_scores'].extend(
                primary_rows(qresponse, ref_pred, variance, anatomy[te], ref_meta))
            ref_thin_assignments = {ref_names[name]: source_thin_assignments[ref_names[name]]
                                    for name in ref_names}
            ref_thin_pred = assignment_predictions(ref_thin_assignments, ref_decoders) \
                if ref_thin_assignments else {}
            frames['source_reference_thin_scores'].extend(
                primary_rows(qresponse, ref_thin_pred, variance, anatomy[te], ref_meta))
            frames['source_reference_selected_scores'].extend(
                selected_rows(qresponse, ref_pred, response, sa, anatomy[te], panel,
                              gene_to_j, ref_meta))
            for source_method, method in ref_names.items():
                z = source_axes[source_method]
                for sid, value, thin_value, lab in zip(
                        query_ids, source_assignments[method]['z_mean'],
                        source_thin_assignments[method]['z_mean'], anatomy[te]):
                    frames['source_reference_positions'].append(dict(
                        **ref_meta, method=method, structure_id=sid, z=float(value),
                        thinned_z=float(thin_value),
                        segment=['S1', 'S2', 'S3'][int(lab)]))

            # Per-regime pooled fits see identical expression rows across algorithm arms.
            for regime in ('full', 'S3_only'):
                cal = calibration if regime == 'full' else calibration & (anatomy[te] == 2)
                meta = dict(**key, regime=regime)
                if not cal.any():
                    frames['failures'].append(dict(**key, stage='pooled', regime=regime,
                                                   method='calibration_population', error='empty'))
                    add_diagnostic(key, 'pooled', regime, 'calibration_population', 'failed',
                                   error='empty')
                    continue
                P = np.vstack([Y, qY[cal]])
                pa = np.r_[sa, np.full(int(cal.sum()), -1, dtype=int)]
                anchors = np.r_[np.ones(len(Y), dtype=bool),
                                np.zeros(int(cal.sum()), dtype=bool)]
                evalY = qY_eval
                evalThinY = thin_qY_eval
                assignments, thin_assignments, decoders = {}, {}, {}
                for algorithm in ('DPT', 'scFates', 'anatomy'):
                    method = ('pooled_anatomy_curve_neighbor' if algorithm == 'anatomy'
                              else 'pooled_' + algorithm + '_neighbor')
                    try:
                        if algorithm == 'DPT':
                            z, info = fit_dpt_axis(P, pa, anchor_mask=anchors,
                                                  n_neighbors=30, seed=SEED)
                        else:
                            z, info = fit_scfates_axis(
                                P, pa, anchor_mask=anchors, nodes=30, seed=SEED,
                                initialization='anatomy' if algorithm == 'anatomy' else 'default')
                        decoder = fit_gene_spline(response, z[:len(Y)], specimens[tr])
                        assignment = project_neighbor_axis(P, z, evalY, k=15, return_distribution=True)
                        thin_assignment = project_neighbor_axis(P, z, evalThinY, k=15, return_distribution=True)
                        decoders[method] = decoder
                        assignments[method] = assignment
                        thin_assignments[method] = thin_assignment
                        add_diagnostic(key, 'pooled', regime, method, 'complete',
                                       metadata=info, z=z[:len(Y)], source_anatomy=sa)
                    except Exception as exc:
                        frames['failures'].append(dict(**key, stage='pooled', regime=regime,
                                                       method=method, error=str(exc)))
                        add_diagnostic(key, 'pooled', regime, method, 'failed', error=exc)
                        print('Failed:', source, 'to', target, 'fold', fold, regime,
                              method, str(exc), flush=True)

                # Include source-only graph methods in each regime's common score table.
                for source_method, z in source_axes.items():
                    name = source_method
                    if name in source_decoders:
                        decoder = source_decoders[name]
                        assignments[name] = project_neighbor_axis(
                            Y, z, evalY, k=15, return_distribution=True)
                        thin_assignments[name] = project_neighbor_axis(
                            Y, z, evalThinY, k=15, return_distribution=True)
                        decoders[name] = decoder
                pred = assignment_predictions(assignments, decoders) if assignments else {}
                thinpred = assignment_predictions(thin_assignments, decoders) if thin_assignments else {}
                pred['segment_coverage_oracle'] = coverage_pred_all[evaluation]
                pred['segment_mean_oracle'] = segment_pred_all[evaluation]
                thinpred['segment_coverage_oracle'] = coverage_pred_all[evaluation]
                thinpred['segment_mean_oracle'] = segment_pred_all[evaluation]
                selected = selected_rows(eval_response, pred, response, sa, eval_a,
                                         panel, gene_to_j, meta) if pred else []
                frames['scores'].extend(primary_rows(eval_response, pred, variance, eval_a, meta))
                frames['thin_scores'].extend(primary_rows(eval_response, thinpred, variance, eval_a, meta))
                frames['selected_scores'].extend(selected)
                frames['thin_selected_scores'].extend(
                    selected_rows(eval_response, thinpred, response, sa, eval_a,
                                  panel, gene_to_j, meta) if thinpred else [])
                for method, assignment in assignments.items():
                    zq = assignment['z_mean']
                    zt = thin_assignments[method]['z_mean']
                    for segment, label in [(None, 'all'), (0, 'S1'), (1, 'S2'), (2, 'S3')]:
                        use = np.ones(len(zq), dtype=bool) if segment is None else eval_a == segment
                        if use.any():
                            frames['stability'].append(dict(
                                **meta, method=method, segment=label, n=int(use.sum()),
                                coordinate_mae=float(np.mean(np.abs(zq[use] - zt[use])))))
                    for sid, value, thin_value, lab in zip(eval_ids, zq, zt, eval_a):
                        frames['positions'].append(dict(
                            **meta, method=method, structure_id=sid, z=float(value),
                            thinned_z=float(thin_value), segment=['S1', 'S2', 'S3'][int(lab)]))
                print('Completed:', source, 'to', target, 'fold', fold, regime, flush=True)

    return {name: pd.DataFrame(rows).to_json(orient='table') for name,rows in frames.items()}


In [ ]:
payload=cached_payload('anatomy_initialization',run_anatomy_initialization,
 root=OUT/'stage_cache',params=CACHE_PARAMS,
 inputs=dict(X=X,counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,anatomy=anatomy,
  markers=marker_mask,gene_fold=gene_fold,n_spots=n_spots,panel=frozen_panel,
  legacy27=legacy27,legacy25=legacy25),code=CODE_DIGEST)
outputs={name:pd.read_json(io.StringIO(str(value)),orient='table') for name,value in payload.items()}
for name,frame in outputs.items():frame.to_csv(OUT/(name+'.csv'),index=False)
display(outputs['diagnostics']);display(outputs['failures'])
# These guards execute after either a fit or a cache hit. Failed new fits stay visible.
def audit_reference(actual,previous,keys,columns,label):
    if actual.duplicated(keys).any() or previous.duplicated(keys).any():
        raise ValueError('Duplicated reference keys: '+label)
    matched=actual.merge(previous,on=keys,validate='one_to_one',suffixes=('_new','_old'))
    if len(matched)!=len(actual) or len(matched)!=len(previous):
        raise ValueError('Reference row/ID support changed: '+label)
    for column in columns:
        if not np.allclose(matched[column+'_new'],matched[column+'_old'],rtol=1e-8,atol=1e-10):
            raise ValueError('Uncached reference reproduction failed: '+label+' '+column)
base27=['scFates_neighbor','DPT_neighbor']
all27=base27+[m+'_assignment' for m in base27]
score_keys=['source_specimen','test_specimen','fold','method','segment']
for current,old in [('source_reference_scores','scores'),('source_reference_thin_scores','thin_scores'),
                    ('source_reference_selected_scores','selected_scores')]:
    audit_reference(outputs[current],legacy27[old][legacy27[old].method.isin(all27)],
                    score_keys,['normalized_mse'],'27 '+old)
position_keys=['source_specimen','test_specimen','fold','method','structure_id']
audit_reference(outputs['source_reference_positions'],legacy27['positions'][legacy27['positions'].method.isin(base27)],
                position_keys,['z','thinned_z'],'27 positions')
base25=['source_scFates_neighbor','source_DPT_neighbor','pooled_scFates_neighbor','pooled_DPT_neighbor']
for name in ['scores','selected_scores','positions']:
    keys=score_keys+['regime'] if name!='positions' else position_keys+['regime']
    columns=['normalized_mse'] if name!='positions' else ['z','thinned_z']
    audit_reference(outputs[name][outputs[name].method.isin(base25)],
                    legacy25[name][legacy25[name].method.isin(base25)],keys,columns,'25 '+name)
diag=outputs['diagnostics']
if len(diag)!=54 or not (diag[diag.method.isin(base25)].status=='complete').all():
    raise RuntimeError('Incomplete baseline attempts: inspect failures before interpreting candidates')
print('Completed: uncached 25/27 score, assignment, ID, coordinate and read reproduction guards')
display(diag.groupby(['stage','method','status']).size().rename('attempt_count').reset_index())
if 'source_medians_ordered' in diag:
    display(diag[diag.status=='complete'].groupby(['stage','method']).source_medians_ordered.agg(['sum','count']))
keys=['source_specimen','test_specimen','fold','regime','segment']
pairs=[]
for suffix in ['', '_assignment']:
    for candidate,baselines in [
       ('source_anatomy_curve_neighbor',['source_scFates_neighbor','source_DPT_neighbor']),
       ('pooled_anatomy_curve_neighbor',['pooled_scFates_neighbor','pooled_DPT_neighbor','source_scFates_neighbor','source_anatomy_curve_neighbor']),
       ('pooled_scFates_neighbor',['source_scFates_neighbor'])]:
        pairs.extend([(candidate+suffix,baseline+suffix) for baseline in baselines])
    pairs.extend([('source_anatomy_curve_neighbor'+suffix,b) for b in ['segment_mean_oracle','segment_coverage_oracle']])
for name in ['scores','thin_scores','selected_scores','thin_selected_scores']:
    frame=outputs[name];parts=[]
    for candidate,baseline in pairs:
        left=frame[frame.method==candidate][keys+['normalized_mse']]
        right=frame[frame.method==baseline][keys+['normalized_mse']]
        match=left.merge(right,on=keys,validate='one_to_one',suffixes=('_candidate','_baseline'))
        match['candidate']=candidate;match['baseline']=baseline
        match['gain_percent']=100*(1-match.normalized_mse_candidate/match.normalized_mse_baseline)
        parts.append(match)
    matched=pd.concat(parts,ignore_index=True)
    matched.to_csv(OUT/(name+'_matched.csv'),index=False)
    summary=matched.groupby(['source_specimen','regime','segment','candidate','baseline']).agg(
        gain_percent=('gain_percent','mean'),fold_count=('fold','nunique')).reset_index()
    summary.to_csv(OUT/(name+'_summary.csv'),index=False)
    display(summary)
# Gene-fold rank agreement is stability, not anatomical truth; each pair uses exactly the same IDs.
positions=outputs['positions'];rank_rows=[]
for key,frame in positions.groupby(['source_specimen','test_specimen','regime','method','segment']):
    wide=frame.pivot(index='structure_id',columns='fold',values='z')
    for i in range(N_FOLDS):
        for j in range(i+1,N_FOLDS):
            if i not in wide or j not in wide:continue
            paired=wide[[i,j]].dropna()
            rank_rows.append(dict(zip(['source_specimen','test_specimen','regime','method','segment'],key),
              fold_a=i,fold_b=j,n=len(paired),spearman=float(spearmanr(paired[i],paired[j]).statistic)))
rank_agreement=pd.DataFrame(rank_rows);rank_agreement.to_csv(OUT/'gene_fold_rank_agreement.csv',index=False)
if not rank_agreement.empty:
    display(rank_agreement.groupby(['source_specimen','regime','method','segment']).agg(
      spearman=('spearman','mean'),pair_count=('fold_a','size')))
calibration_sensitivity=positions[positions.regime=='full'].merge(
 positions[positions.regime=='S3_only'],on=position_keys,validate='one_to_one',suffixes=('_full','_partial'))
calibration_sensitivity['coordinate_change']=abs(calibration_sensitivity.z_full-calibration_sensitivity.z_partial)
calibration_sensitivity=calibration_sensitivity.groupby(['source_specimen','method']).agg(
 coordinate_mae=('coordinate_change','mean'),n=('structure_id','size')).reset_index()
calibration_sensitivity.to_csv(OUT/'calibration_sensitivity.csv',index=False);display(calibration_sensitivity)
# Development-selected programs have selection bias and no independent physical position ground truth.
import matplotlib.pyplot as plt
program=pd.read_csv(OUT/'selected_scores_matched.csv')
fig,axes=plt.subplots(2,2,figsize=(11,7),sharey=True)
plot_methods=[('source_anatomy_curve_neighbor_assignment','Source anatomy'),
 ('pooled_anatomy_curve_neighbor_assignment','Pooled anatomy'),
 ('pooled_scFates_neighbor_assignment','Pooled generic')]
for row,source in enumerate(CONTROLS):
    for col,regime in enumerate(['full','S3_only']):
        ax=axes[row,col]
        for offset,color,(candidate,label) in zip([-.2,0,.2],['#0072B2','#D55E00','#009E73'],plot_methods):
            sub=program[(program.source_specimen==source)&(program.regime==regime)&
                (program.candidate==candidate)&(program.baseline=='source_scFates_neighbor_assignment')]
            for seg,name in enumerate(['S1_selected','S2_selected','S3_selected']):
                vals=sub[sub.segment==name].gain_percent
                if len(vals):
                    ax.scatter(np.repeat(seg+offset,len(vals)),vals,s=18,alpha=.65,color=color,label=label if seg==0 else None)
                    ax.plot([seg+offset-.06,seg+offset+.06],[vals.mean()]*2,color='black',lw=1)
        ax.axhline(0,color='gray',lw=.8);ax.set_xticks([0,1,2],['S1','S2','S3'])
        ax.set_title(source+' → held-out control; '+regime)
        if col==0:ax.set_ylabel('Program error reduction (%)')
        if row==1:ax.set_xlabel('Within-segment development program')
axes[0,0].legend(fontsize=8);fig.tight_layout();fig.savefig(OUT/'anatomy_initialization_program_gains.png',dpi=160)
display(fig);plt.close(fig)


## Measured decision and next goal checkpoint

All 54 attempts completed, including all 18 anatomy-initialized curves. All
source S1/S2/S3 medians remain ordered and all notebook-25/27 reference guards
pass. These technical checks do not establish useful fine reconstruction.

Source anatomy initialization changes assignment-aware all-gene error by
−0.014%/+0.003% in the two transfer directions. Selected S1/S2/S3 program gains
versus generic source scFates are +0.12%/+0.86%/−0.74% and
+0.14%/+0.06%/−0.08%, with three folds per cell. Pooled full-calibration gains
are +0.55%/+1.79%/−1.23% and +0.97%/−10.02%/−0.07%. The ordered starting
path does not resolve the reverse S2 prediction failure.

Source S3 gene-fold rank agreement drops 0.943→0.818 in the first direction.
Reverse full-calibration pooled S2 agreement improves 0.447→0.924, while its
held-out program prediction worsens. Reverse partial-calibration pooled S3
agreement falls 0.235→0.089. Rank stability and ordered segment medians must
therefore be reported beside prediction rather than substituted for it.

**Do not adopt or expand this initialization branch.** It is an existing-method
component, not a distinctive estimator; this run rejects its usefulness rather
than the repeated-structure framework. Before another experiment, identify how
the ordered physical-object prior enters fitting and how the estimator protects
canonical position against specimen and sampling effects. No endpoint-only
ablation, parameter search or downstream injury/regulator analysis follows this
result.

Only two controls support descriptive development. Frozen program panels were
selected during prior method development and are not independent positional
ground truth. Shared all-gene normalization/exposure also couples gene folds.
Upstream source anatomy inherits marker information. Neighbor assignments are
not calibrated anatomical posteriors, gene-fold rank agreement is stability, and
curve arc length is not physical nephron distance. Depth is excluded throughout.
